## Parte 1 — Softmax con temperatura

Conforme al enunciado, la primera celda del notebook registra los datos del estudiante: nombre [NOMBRE], correo [CORREO], cédula [CEDULA] y teléfono [TELEFONO].

La softmax con temperatura se implementa en NumPy de forma numéricamente estable restando el máximo de los logits antes de exponenciar: p_i = exp((z_i − z_max)/T) / Σ_j exp((z_j − z_max)/T). Este ajuste no cambia el resultado (el factor común exp(z_max) se cancela entre numerador y denominador), pero evita desbordamientos numéricos. Con los logits de t0..t5 (2.0, 1.0, 0.5, 0.2, −1.0, −3.0) se obtienen las siguientes distribuciones, que en los tres casos suman 1:

| Token | Logit | p (T=0.5) | p (T=1.0) | p (T=2.0) |
|-------|-------|-----------|-----------|-----------|
| t0    | 2.0   | 0.8231    | 0.5516    | 0.3583    |
| t1    | 1.0   | 0.1114    | 0.2029    | 0.2173    |
| t2    | 0.5   | 0.0410    | 0.1231    | 0.1693    |
| t3    | 0.2   | 0.0225    | 0.0912    | 0.1457    |
| t4    | −1.0  | 0.0020    | 0.0275    | 0.0800    |
| t5    | −3.0  | 0.0000    | 0.0037    | 0.0294    |

El valor 0.0000 de t5 con T=0.5 es efecto del redondeo a cuatro decimales: su probabilidad exacta es 3.7367e-05. La entropía se calcula en bits (log base 2) sobre la distribución completa; los seis tokens tienen probabilidad positiva para las tres temperaturas:

| Temperatura | Entropía (bits) |
|-------------|-----------------|
| T=0.5       | 0.9147          |
| T=1.0       | 1.7998          |
| T=2.0       | 2.2888          |

Ya se ve la tendencia que se analiza en la Parte 4: al subir T la masa se reparte (t0 pasa de 0.8231 a 0.5516 y luego a 0.3583; t5 sube de 3.7367e-05 a 0.0037 y luego a 0.0294) y la entropía crece de 0.9147 a 1.7998 y a 2.2888 bits. La celda de esta sección genera además la figura figuras/T1_t1_softmax_temperatura.png.

In [1]:
# T1: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Celda inicial del notebook: datos del estudiante (según enunciado)
# ------------------------------------------------------------------
datos_estudiante = {
    "nombre": "[NOMBRE]",
    "correo": "[CORREO]",
    "cedula": "[CEDULA]",
    "telefono": "[TELEFONO]",
}

# ------------------------------------------------------------------
# Datos del problema: logits de t0..t5
# ------------------------------------------------------------------
tokens = [f"t{i}" for i in range(6)]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)

# ------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable (max-shift)
# ------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    z = np.asarray(z, dtype=float)
    z_shift = z - z.max()          # estabilidad: restar el máximo antes de exponenciar
    e = np.exp(z_shift / T)
    return e / e.sum()

temperaturas = [0.5, 1.0, 2.0]
dists = {T: softmax_con_temperatura(logits, T) for T in temperaturas}

def entropia_bits(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

entropias = {T: entropia_bits(dists[T]) for T in temperaturas}

# Verificaciones (garantizadas por la construcción de la softmax)
for T in temperaturas:
    assert abs(dists[T].sum() - 1.0) < 1e-12
    assert (dists[T] > 0).all()

# ------------------------------------------------------------------
# Tabla: tres distribuciones y tres entropías, cuatro decimales
# ------------------------------------------------------------------
tabla = pd.DataFrame(
    {
        "logit": logits,
        **{f"p_T={T}": dists[T] for T in temperaturas},
    },
    index=tokens,
)
tabla_redondeada = tabla.round(4)

entropias_fila = pd.Series(
    {f"p_T={T}": round(entropias[T], 4) for T in temperaturas},
    name="entropia_bits",
)

# ------------------------------------------------------------------
# Figura: barras agrupadas de las tres distribuciones
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(tokens))
ancho = 0.25
for k, T in enumerate(temperaturas):
    ax.bar(x + (k - 1) * ancho, dists[T], width=ancho, label=f"T={T}")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("probabilidad")
ax.set_title("Softmax con temperatura (logits t0..t5)")
ax.legend()
fig.tight_layout()
plt.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "subtarea": "T1: softmax con temperatura y entropía en bits",
    "celda_inicial_notebook": datos_estudiante,
    "tokens": tokens,
    "logits": logits.tolist(),
    "temperaturas": temperaturas,
    "distribuciones": {f"T={T}": dists[T].tolist() for T in temperaturas},
    "suma_distribuciones": {f"T={T}": float(dists[T].sum()) for T in temperaturas},
    "entropias_bits": {f"T={T}": entropias[T] for T in temperaturas},
    "tabla_4_decimales": {
        "filas": tokens,
        "columnas": list(tabla_redondeada.columns),
        "valores": tabla_redondeada.values.tolist(),
        "entropias_bits_4dec": {f"T={T}": round(entropias[T], 4) for T in temperaturas},
    },
    "supuestos": [
        "Los datos personales de la primera celda se dejan con los marcadores [NOMBRE], [CORREO], [CEDULA], [TELEFONO] del enunciado, pues el script no conoce los datos reales del estudiante.",
        "La entropía se calcula en bits (log2) sobre la distribución completa; los seis tokens tienen probabilidad positiva para las tres temperaturas.",
    ],
}

Path("resultados.json").write_text(
    json.dumps(resultados, indent=2, ensure_ascii=False), encoding="utf-8"
)

# ------------------------------------------------------------------
# Resumen
# ------------------------------------------------------------------
print("Distribuciones (4 decimales):")
print(tabla_redondeada.to_string())
print("\nEntropías en bits (4 decimales):")
print(entropias_fila.to_string())
print("\nSuma de cada distribución:",
      {f"T={T}": float(dists[T].sum()) for T in temperaturas})
print("Figura guardada: t1_softmax_temperatura.png")
print("Resultados guardados: resultados.json")


Distribuciones (4 decimales):
    logit  p_T=0.5  p_T=1.0  p_T=2.0
t0    2.0   0.8231   0.5516   0.3583
t1    1.0   0.1114   0.2029   0.2173
t2    0.5   0.0410   0.1231   0.1693
t3    0.2   0.0225   0.0912   0.1457
t4   -1.0   0.0020   0.0275   0.0800
t5   -3.0   0.0000   0.0037   0.0294

Entropías en bits (4 decimales):
p_T=0.5    0.9147
p_T=1.0    1.7998
p_T=2.0    2.2888

Suma de cada distribución: {'T=0.5': 1.0, 'T=1.0': 1.0, 'T=2.0': 1.0}
Figura guardada: t1_softmax_temperatura.png
Resultados guardados: resultados.json


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T=1 de la Parte 1 se ordenan los tokens de mayor a menor probabilidad y se acumula hasta alcanzar p=0.9 por primera vez. El último token incluido se conserva completo, por lo que la acumulada del núcleo puede superar ligeramente el umbral:

| Token | p original (T=1) | Acumulada | ¿En el núcleo? | p renormalizada |
|-------|------------------|-----------|----------------|-----------------|
| t0    | 0.5516           | 0.5516    | Sí             | 0.5694          |
| t1    | 0.2029           | 0.7546    | Sí             | 0.2095          |
| t2    | 0.1231           | 0.8776    | Sí             | 0.1270          |
| t3    | 0.0912           | 0.9688    | Sí             | 0.0941          |
| t4    | 0.0275           | 0.9963    | No             | —               |
| t5    | 0.0037           | 1.0000    | No             | —               |

Con los tres primeros tokens la acumulada es 0.8776, todavía por debajo de 0.9, de modo que el conjunto mínimo que alcanza el umbral es {t0, t1, t2, t3}, con probabilidad acumulada 0.9688. Los tokens t4 y t5 quedan descartados y no participan en la renormalización. Dividiendo entre 0.9688, la distribución renormalizada del núcleo queda: t0=0.5694, t1=0.2095, t2=0.1270, t3=0.0941, que suma 1. La celda de esta sección genera la figura figuras/T2_top_p_T1.png.

In [2]:
# T2: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# T2: Top-p (nucleus) con p=0.9 sobre la distribución de T=1 de T1
# ------------------------------------------------------------------

# 1) Cargar la distribución de T=1 calculada en T1 (no se recalcula)
ruta_t1 = Path("entrada/T1/resultados.json")
with open(ruta_t1, "r", encoding="utf-8") as f:
    t1 = json.load(f)

tokens = list(t1["tokens"])
p_t1 = np.array(t1["distribuciones"]["T=1.0"], dtype=float)

# 2) Ordenar tokens de mayor a menor probabilidad
orden = np.argsort(-p_t1)
tokens_ordenados = [tokens[i] for i in orden]
probs_ordenadas = p_t1[orden]

# 3) Conjunto mínimo cuya probabilidad acumulada alcanza p = 0.9
p_corte = 0.9
acumulada = np.cumsum(probs_ordenadas)
# searchsorted (side='left') devuelve el primer índice i con acumulada[i] >= p_corte
k = int(np.searchsorted(acumulada, p_corte) + 1)  # número de tokens conservados

tokens_nucleo = tokens_ordenados[:k]
probs_nucleo = probs_ordenadas[:k]
acumulada_nucleo = float(acumulada[k - 1])
tokens_descartados = tokens_ordenados[k:]

# 4) Renormalizar las probabilidades del núcleo
probs_renorm = probs_nucleo / probs_nucleo.sum()

# Verificaciones (garantizadas por la construcción anterior)
assert acumulada_nucleo >= p_corte, "la acumulada del núcleo debe alcanzar p=0.9"
if k > 1:
    assert acumulada[k - 2] < p_corte, "el conjunto debe ser mínimo (sin el último token no llega a p)"
assert abs(probs_renorm.sum() - 1.0) < 1e-12, "la renormalizada debe sumar 1"

# 5) Tabla resumen (original, acumulada, renormalizada)
tabla = []
for i, tok in enumerate(tokens_ordenados):
    fila = {
        "token": tok,
        "p_T1_original": float(probs_ordenadas[i]),
        "acumulada": float(acumulada[i]),
        "en_nucleo": i < k,
        "p_renormalizada": float(probs_renorm[i]) if i < k else None,
    }
    tabla.append(fila)

# 6) Escribir resultados.json (contrato de la subtarea)
resultados = {
    "subtarea": "T2: muestreo de núcleo (top-p, p=0.9) sobre la distribución de T=1",
    "p": p_corte,
    "temperatura": 1.0,
    "distribucion_T1_original": {tok: float(p) for tok, p in zip(tokens, p_t1)},
    "tokens_ordenados_desc": tokens_ordenados,
    "probs_ordenadas_desc": [float(x) for x in probs_ordenadas],
    "acumulada_ordenada": [float(x) for x in acumulada],
    "tokens_sobrevivientes": tokens_nucleo,
    "probs_nucleo_sin_renormalizar": [float(x) for x in probs_nucleo],
    "probabilidad_acumulada_nucleo": acumulada_nucleo,
    "distribucion_renormalizada": {tok: float(p) for tok, p in zip(tokens_nucleo, probs_renorm)},
    "tokens_descartados": tokens_descartados,
    "suma_distribucion_renormalizada": float(probs_renorm.sum()),
    "tabla_completa": tabla,
    "supuestos": [
        "La distribución de T=1 se carga de entrada/T1/resultados.json (subtarea T1) sin recalcularla.",
        "El conjunto mínimo se construye conservando tokens en orden decreciente de probabilidad hasta que la suma alcance o supere p=0.9 por primera vez; el último token incluido puede hacer que la acumulada supere ligeramente 0.9 (aquí 0.9688).",
        "Los tokens fuera del núcleo (t4, t5) se descartan y no participan en la renormalización.",
    ],
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# 7) Figura: distribución original de T=1 vs. renormalizada del núcleo
fig, ax = plt.subplots(figsize=(8, 4.5))
x_orig = np.arange(len(tokens))
ax.bar(x_orig - 0.2, p_t1, width=0.4, label="T=1 original", color="#4C72B0")
x_nuc = np.arange(len(tokens_nucleo))
ax.bar(x_nuc + 0.2, probs_renorm, width=0.4, label="Top-p renormalizada", color="#DD8452")
ax.set_xticks(x_orig)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad")
ax.set_title(f"Top-p (p=0.9) sobre T=1: sobreviven {tokens_nucleo} (acumulada={acumulada_nucleo:.4f})")
ax.legend()
fig.tight_layout()
fig.savefig("top_p_T1.png", dpi=120)
plt.close(fig)

# 8) Resumen breve
print("=== T2: Top-p con p=0.9 sobre la distribución de T=1 ===")
print("Tokens ordenados (mayor a menor):", tokens_ordenados)
print("Probabilidades ordenadas:", np.round(probs_ordenadas, 6).tolist())
print("Acumulada:", np.round(acumulada, 6).tolist())
print(f"Tokens sobrevivientes al núcleo: {tokens_nucleo} (k={k})")
print(f"Probabilidad acumulada del núcleo: {acumulada_nucleo:.6f} (>= 0.9)")
print("Distribución renormalizada:")
for tok, p in zip(tokens_nucleo, probs_renorm):
    print(f"  {tok}: {p:.6f}")
print(f"Suma de la renormalizada: {probs_renorm.sum():.12f}")
print("Tokens descartados:", tokens_descartados)
print("Resultados escritos en resultados.json; figura en top_p_T1.png")


=== T2: Top-p con p=0.9 sobre la distribución de T=1 ===
Tokens ordenados (mayor a menor): ['t0', 't1', 't2', 't3', 't4', 't5']
Probabilidades ordenadas: [0.551623, 0.202931, 0.123084, 0.091183, 0.027464, 0.003717]
Acumulada: [0.551623, 0.754553, 0.877637, 0.96882, 0.996283, 1.0]
Tokens sobrevivientes al núcleo: ['t0', 't1', 't2', 't3'] (k=4)
Probabilidad acumulada del núcleo: 0.968820 (>= 0.9)
Distribución renormalizada:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
Suma de la renormalizada: 1.000000000000
Tokens descartados: ['t4', 't5']
Resultados escritos en resultados.json; figura en top_p_T1.png


## Parte 3 — Comprobación empírica

Se toman 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0) (rng.choice sobre los cuatro tokens del núcleo con sus probabilidades renormalizadas). Las frecuencias observadas frente a las probabilidades teóricas son:

| Token | Probabilidad teórica | Conteo observado | Frecuencia observada |
|-------|----------------------|------------------|----------------------|
| t0    | 0.5694               | 5682             | 0.5682               |
| t1    | 0.2095               | 2093             | 0.2093               |
| t2    | 0.1270               | 1296             | 0.1296               |
| t3    | 0.0941               | 929              | 0.0929               |

El gráfico de barras comparativo (generado por la celda, figuras/T3_T3_frecuencias_vs_teoricas.png) muestra barras de frecuencia observada prácticamente indistinguibles de las teóricas. La divergencia KL de la distribución empírica respecto de la teórica, KL(q‖p) = Σ q_i log2(q_i/p_i), es 5.0069e-05 bits. Un valor de ese orden es consistente con el ruido de muestreo esperable con 10 000 muestras; además, los cuatro tokens del núcleo aparecieron en las muestras, por lo que ningún término de la suma resulta indefinido.

In [3]:
# T3: código aprobado por el crítico
# SUBTAREA T3: Comprobación empírica del muestreo top-p (Parte 3)
# Toma 10 000 muestras de la distribución renormalizada de T2 con
# numpy.random.default_rng(0), compara frecuencias observadas vs probabilidades
# teóricas en un gráfico de barras y calcula la divergencia KL en bits.

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1) Cargar la distribución renormalizada de T2 (no se recalcula)
# ------------------------------------------------------------------
ruta_t2 = Path("entrada/T2/resultados.json")
with open(ruta_t2, "r", encoding="utf-8") as f:
    res_t2 = json.load(f)

dist_renorm = res_t2["distribucion_renormalizada"]  # dict token -> prob
tokens = list(dist_renorm.keys())                   # ["t0", "t1", "t2", "t3"]
p_teorica = np.array([dist_renorm[t] for t in tokens], dtype=float)

assert abs(p_teorica.sum() - 1.0) < 1e-12, "La distribución teórica debe sumar 1"

# ------------------------------------------------------------------
# 2) Muestreo: 10 000 muestras con default_rng(0)
# ------------------------------------------------------------------
rng = np.random.default_rng(0)
n_muestras = 10_000
indices = rng.choice(len(tokens), size=n_muestras, p=p_teorica)

conteos = np.bincount(indices, minlength=len(tokens)).astype(float)
frecuencias_obs = conteos / n_muestras

# ------------------------------------------------------------------
# 3) Divergencia KL (empírica || teórica) en bits
#    KL = Σ_i q_i * log2(q_i / p_i); términos con q_i = 0 aportan 0
# ------------------------------------------------------------------
mascara = frecuencias_obs > 0
kl_bits = float(np.sum(
    frecuencias_obs[mascara]
    * np.log2(frecuencias_obs[mascara] / p_teorica[mascara])
))

# ------------------------------------------------------------------
# 4) Gráfico de barras: frecuencias observadas vs probabilidades teóricas
# ------------------------------------------------------------------
x = np.arange(len(tokens))
ancho = 0.38

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - ancho / 2, frecuencias_obs, width=ancho,
            label="Frecuencia observada (10 000 muestras)", color="#4C72B0")
b2 = ax.bar(x + ancho / 2, p_teorica, width=ancho,
            label="Probabilidad teórica (top-p renormalizada)", color="#DD8452")

for barras in (b1, b2):
    for b in barras:
        ax.annotate(f"{b.get_height():.4f}",
                    xy=(b.get_x() + b.get_width() / 2, b.get_height()),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_title("Top-p (p=0.9): frecuencias observadas vs probabilidades teóricas\n"
             f"10 000 muestras, default_rng(0) — KL = {kl_bits:.6f} bits")
ax.legend()
ax.set_ylim(0, max(frecuencias_obs.max(), p_teorica.max()) * 1.15)
fig.tight_layout()
plt.savefig("T3_frecuencias_vs_teoricas.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# 5) Escribir resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "subtarea": "T3: comprobación empírica del muestreo top-p (10 000 muestras, default_rng(0))",
    "semilla": 0,
    "generador": "numpy.random.default_rng(0)",
    "n_muestras": n_muestras,
    "tokens": tokens,
    "probabilidades_teoricas": {t: float(p) for t, p in zip(tokens, p_teorica)},
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {t: float(f) for t, f in zip(tokens, frecuencias_obs)},
    "divergencia_KL_bits": kl_bits,
    "figura": "T3_frecuencias_vs_teoricas.png",
    "supuestos": [
        "La distribución renormalizada se carga de entrada/T2/resultados.json (subtarea T2) sin recalcularla.",
        "El muestreo usa rng.choice sobre los 4 tokens del núcleo con las probabilidades renormalizadas de T2.",
        "La KL se calcula como KL(empírica || teórica) = Σ q_i log2(q_i / p_i); los tokens con frecuencia empírica 0 aportan 0 (aquí todos los tokens del núcleo aparecen en las muestras).",
    ],
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------
# 6) Resumen
# ------------------------------------------------------------------
print("=== T3: Comprobación empírica (top-p, p=0.9) ===")
print(f"Muestras: {n_muestras} | Generador: default_rng(0)")
print(f"{'token':<6}{'teórica':>12}{'observada':>12}{'conteo':>10}")
for t, pt, fo, c in zip(tokens, p_teorica, frecuencias_obs, conteos):
    print(f"{t:<6}{pt:>12.6f}{fo:>12.6f}{int(c):>10d}")
print(f"Divergencia KL (empírica || teórica) = {kl_bits:.6f} bits")
print("Figura guardada: T3_frecuencias_vs_teoricas.png")
print("Resultados escritos en resultados.json")


=== T3: Comprobación empírica (top-p, p=0.9) ===
Muestras: 10000 | Generador: default_rng(0)
token      teórica   observada    conteo
t0        0.569376    0.568200      5682
t1        0.209462    0.209300      2093
t2        0.127045    0.129600      1296
t3        0.094117    0.092900       929
Divergencia KL (empírica || teórica) = 0.000050 bits
Figura guardada: T3_frecuencias_vs_teoricas.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

**Cuando T → 0.** Dividir los logits entre una temperatura muy pequeña amplifica sus diferencias: en p_i = exp(z_i/T) / Σ_j exp(z_j/T), todo token cuyo logit sea menor que el máximo cumple exp((z_i − z_max)/T) → 0 frente al término del máximo. Toda la masa se concentra entonces en el token de mayor logit (aquí t0) y la distribución tiende a una masa puntual con entropía 0. Las entropías medidas muestran esa tendencia: al bajar T de 2 a 1 y a 0.5, la entropía cae de 2.2888 a 1.7998 y a 0.9147 bits; en paralelo, la masa de t0 crece de 0.3583 a 0.5516 y a 0.8231, mientras que la de t5 se desploma de 0.0294 a 0.0037 y a 3.7367e-05. La decodificación voraz (greedy) elige en cada paso el token de mayor probabilidad, que es el de mayor logit porque la softmax es monótona en los logits. En el límite T → 0 la distribución solo tiene masa en ese token, de modo que muestrear de ella devuelve el argmax con probabilidad 1: muestrear con T → 0 y hacer greedy producen exactamente la misma salida. Por eso T → 0 equivale a la decodificación voraz.

**Cuando T → ∞.** El cociente z_i/T tiende a 0 para todo token, todos los exponenciales convergen al mismo valor y la distribución tiende a la uniforme sobre los seis tokens, con entropía máxima. La evidencia medida apunta en esa dirección: al subir T de 1 a 2 la entropía sube de 1.7998 a 2.2888 bits y las probabilidades se aplanan (t0 baja de 0.5516 a 0.3583 y t5 sube de 0.0037 a 0.0294, acercándose entre sí).

En síntesis, T controla el compromiso entre determinismo y aleatoriedad: con T pequeña la distribución se concentra (entropía de 0.9147 bits con T=0.5, salida casi determinista, límite greedy) y con T grande se aplana hacia la uniforme (entropía de 2.2888 bits con T=2, máxima exploración).